In [79]:
import pandas as pd
import plotly.express as px
from django.contrib.admin import display
from sqlalchemy               import (create_engine)
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import os
from sklearn.model_selection  import train_test_split
from sklearn.ensemble         import RandomForestClassifier
from sklearn.metrics          import classification_report,roc_auc_score, confusion_matrix,average_precision_score,accuracy_score,precision_score,recall_score,f1_score
from sklearn.preprocessing    import StandardScaler
from sklearn.linear_model     import LogisticRegression
from sklearn.naive_bayes      import BernoulliNB
from xgboost                  import XGBClassifier

In [2]:
outpatient_path = r"C:\Users\Νίκος\Documents\N'work\thesis\Git_Repository\medicare-data-mining\data\raw\2015_2025\outpatient.csv"

outpatient_df = pd.read_csv(outpatient_path, sep="|", low_memory=False)

print(outpatient_df.shape)
print(outpatient_df.columns[:20])

(575092, 162)
Index(['BENE_ID', 'CLM_ID', 'NCH_NEAR_LINE_REC_IDENT_CD', 'NCH_CLM_TYPE_CD',
       'CLM_FROM_DT', 'CLM_THRU_DT', 'NCH_WKLY_PROC_DT', 'FI_CLM_PROC_DT',
       'CLAIM_QUERY_CODE', 'PRVDR_NUM', 'CLM_FAC_TYPE_CD',
       'CLM_SRVC_CLSFCTN_TYPE_CD', 'CLM_FREQ_CD', 'FI_NUM',
       'CLM_MDCR_NON_PMT_RSN_CD', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT',
       'NCH_PRMRY_PYR_CD', 'PRVDR_STATE_CD', 'ORG_NPI_NUM'],
      dtype='str')


In [27]:
outpatient_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 575092 entries, 0 to 575091
Columns: 163 entries, bene_id to year
dtypes: datetime64[us](1), float64(36), int32(1), int64(26), str(99)
memory usage: 713.0 MB


In [28]:
for col in outpatient_df.columns:
    print(col)

bene_id
clm_id
nch_near_line_rec_ident_cd
nch_clm_type_cd
clm_from_dt
clm_thru_dt
nch_wkly_proc_dt
fi_clm_proc_dt
claim_query_code
prvdr_num
clm_fac_type_cd
clm_srvc_clsfctn_type_cd
clm_freq_cd
fi_num
clm_mdcr_non_pmt_rsn_cd
clm_pmt_amt
nch_prmry_pyr_clm_pd_amt
nch_prmry_pyr_cd
prvdr_state_cd
org_npi_num
at_physn_upin
at_physn_npi
op_physn_upin
op_physn_npi
ot_physn_upin
ot_physn_npi
clm_mco_pd_sw
ptnt_dschrg_stus_cd
clm_tot_chrg_amt
nch_bene_blood_ddctbl_lblty_am
nch_profnl_cmpnt_chrg_amt
prncpal_dgns_cd
icd_dgns_cd1
icd_dgns_cd2
icd_dgns_cd3
icd_dgns_cd4
icd_dgns_cd5
icd_dgns_cd6
icd_dgns_cd7
icd_dgns_cd8
icd_dgns_cd9
icd_dgns_cd10
icd_dgns_cd11
icd_dgns_cd12
icd_dgns_cd13
icd_dgns_cd14
icd_dgns_cd15
icd_dgns_cd16
icd_dgns_cd17
icd_dgns_cd18
icd_dgns_cd19
icd_dgns_cd20
icd_dgns_cd21
icd_dgns_cd22
icd_dgns_cd23
icd_dgns_cd24
icd_dgns_cd25
fst_dgns_e_cd
icd_dgns_e_cd1
icd_dgns_e_cd2
icd_dgns_e_cd3
icd_dgns_e_cd4
icd_dgns_e_cd5
icd_dgns_e_cd6
icd_dgns_e_cd7
icd_dgns_e_cd8
icd_dgns_e_cd9

In [29]:
outpatient_df["clm_pmt_amt"] = pd.to_numeric(
    outpatient_df["clm_pmt_amt"],
    errors="coerce"
)

In [30]:
outpatient_df["clm_pmt_amt"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

count    575092.000000
mean       2374.189815
std        8425.516567
min          59.640000
50%        1003.705000
75%        1355.652500
90%        1958.090000
95%        9289.152000
99%       45283.600000
max      428631.860000
Name: clm_pmt_amt, dtype: float64

In [36]:
diagnosis_cols = [
    col for col in outpatient_df.columns
    if col.startswith("icd_dgns_cd") or col == "prncpal_dgns_cd"
]

In [37]:
def has_prefix(row, prefixes):
    return any(
        str(code).startswith(prefix)
        for code in row
        if pd.notnull(code)
        for prefix in prefixes
    )

outpatient_df["diabetes"] = outpatient_df[diagnosis_cols].apply(
    lambda row: has_prefix(row, ["E08", "E09", "E10", "E11", "E13"]),
    axis=1
).astype(int)

outpatient_df["ckd"] = outpatient_df[diagnosis_cols].apply(
    lambda row: has_prefix(row, ["N18"]),
    axis=1
).astype(int)

outpatient_df["chf"] = outpatient_df[diagnosis_cols].apply(
    lambda row: has_prefix(row, ["I50"]),
    axis=1
).astype(int)

outpatient_df["copd"] = outpatient_df[diagnosis_cols].apply(
    lambda row: has_prefix(row, ["J44"]),
    axis=1
).astype(int)

outpatient_df["cancer"] = outpatient_df[diagnosis_cols].apply(
    lambda row: has_prefix(row, ["C"]),
    axis=1
).astype(int)

outpatient_df["stroke"] = outpatient_df[diagnosis_cols].apply(
    lambda row: has_prefix(row, ["I63"]),
    axis=1
).astype(int)

In [39]:
patient_year_df = (
    outpatient_df
        .groupby(["bene_id", "year"])
        .agg(
            total_annual_cost=("clm_pmt_amt", "sum"),
            claim_count=("clm_id", "nunique"),
            line_count=("clm_line_num", "count"),
            diabetes=("diabetes", "max"),
            ckd=("ckd", "max"),
            chf=("chf", "max"),
            copd=("copd", "max"),
            cancer=("cancer", "max"),
            stroke=("stroke", "max")
        )
        .reset_index()
)

In [40]:
condition_cols = [
    "diabetes",
    "ckd",
    "chf",
    "copd",
    "cancer",
    "stroke"
]

patient_year_df["comorbidity_count"] = (
    patient_year_df[condition_cols].sum(axis=1)
)

In [41]:
threshold = patient_year_df["total_annual_cost"].quantile(0.90)

patient_year_df["high_cost"] = (
    patient_year_df["total_annual_cost"] > threshold
).astype(int)

In [33]:
patient_year_df = (
    outpatient_df
        .groupby(["bene_id", "year"])
        .agg(
            total_annual_cost=("clm_pmt_amt", "sum"),
            claim_count=("clm_id", "nunique"),
            line_count=("clm_line_num", "count")
        )
        .reset_index()
)

In [42]:
feature_cols = [
    "claim_count",
    "line_count",
    "diabetes",
    "ckd",
    "chf",
    "copd",
    "cancer",
    "stroke",
    "comorbidity_count"
]

In [43]:
# σιγουρευόμαστε ότι το cost είναι numeric
outpatient_df["clm_pmt_amt"] = pd.to_numeric(outpatient_df["clm_pmt_amt"], errors="coerce").fillna(0)

# conditions που έφτιαξες
condition_cols = ["diabetes", "ckd", "chf", "copd", "cancer", "stroke"]

# patient-year dataset
patient_year_df = (
    outpatient_df
    .groupby(["bene_id", "year"])
    .agg(
        total_annual_cost=("clm_pmt_amt", "sum"),
        claim_count=("clm_id", "nunique"),
        line_count=("clm_line_num", "count"),
        diabetes=("diabetes", "max"),
        ckd=("ckd", "max"),
        chf=("chf", "max"),
        copd=("copd", "max"),
        cancer=("cancer", "max"),
        stroke=("stroke", "max"),
    )
    .reset_index()
)

# comorbidity count
patient_year_df["comorbidity_count"] = patient_year_df[condition_cols].sum(axis=1)

# target: top 10% by annual cost
threshold = patient_year_df["total_annual_cost"].quantile(0.90)
patient_year_df["high_cost"] = (patient_year_df["total_annual_cost"] > threshold).astype(int)

patient_year_df[["total_annual_cost","high_cost"]].describe(), patient_year_df["high_cost"].value_counts(normalize=True)

(       total_annual_cost     high_cost
 count       4.435100e+04  44351.000000
 mean        3.078572e+04      0.099998
 std         1.185857e+05      0.300000
 min         5.964000e+01      0.000000
 25%         3.696100e+02      0.000000
 50%         6.227350e+03      0.000000
 75%         2.130351e+04      0.000000
 max         9.279700e+06      1.000000,
 high_cost
 0    0.900002
 1    0.099998
 Name: proportion, dtype: float64)

In [44]:
# Skew check
patient_year_df["total_annual_cost"].describe(percentiles=[.5,.75,.9,.95,.99])

# Group comparison
patient_year_df.groupby("high_cost")[["claim_count","line_count","comorbidity_count"]].mean()

,claim_count,line_count,comorbidity_count
high_cost,,,
0,4.007992,7.323705,0.731386
1,54.717024,63.756257,1.346110


In [52]:
patient_year_df = patient_year_df.sort_values(["bene_id", "year"])

# shift target προς τα πίσω (δηλαδή το cost του επόμενου έτους)
patient_year_df["high_cost_next_year"] = (
    patient_year_df
    .groupby("bene_id")["high_cost"]
    .shift(-1)
)

In [53]:
patient_year_df = patient_year_df.dropna(
    subset=["high_cost_next_year"]
)

In [54]:
train_df = patient_year_df[
    patient_year_df["year"].between(2015, 2019)
]

test_df = patient_year_df[
    patient_year_df["year"].between(2020, 2025)
]

In [55]:
feature_cols = [
    "claim_count",
    "line_count",
    "diabetes",
    "ckd",
    "chf",
    "copd",
    "cancer",
    "stroke",
    "comorbidity_count"
]

X_train = train_df[feature_cols]
y_train = train_df["high_cost_next_year"]

X_test = test_df[feature_cols]
y_test = test_df["high_cost_next_year"]

In [57]:
continuous_cols = ["claim_count", "line_count", "comorbidity_count"]

scaler = StandardScaler()

X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

In [58]:
model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [59]:
y_prob = model.predict_proba(X_test)[:,1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))

ROC-AUC: 0.8034109810072856
PR-AUC: 0.40968407747223534


In [72]:
model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

In [81]:
# 1. Probabilities
y_prob = model.predict_proba(X_test)[:, 1]

# 2. Binary predictions
y_pred = (y_prob >= 0.5).astype(int)

# 3. Metrics that need probabilities
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

ROC-AUC: 0.8034109810072856
PR-AUC: 0.40968407747223534
[[10388  1735]
 [  501   822]]
              precision    recall  f1-score   support

         0.0       0.95      0.86      0.90     12123
         1.0       0.32      0.62      0.42      1323

    accuracy                           0.83     13446
   macro avg       0.64      0.74      0.66     13446
weighted avg       0.89      0.83      0.86     13446



Το τελικό μοντέλο λογιστικής παλινδρόμησης, εκπαιδευμένο με χρονικό διαχωρισμό (2015–2019 training και 2020–2025 testing) και πρόβλεψη υψηλού κόστους για το επόμενο έτος, παρουσίασε ικανοποιητική και ρεαλιστική προγνωστική απόδοση. Συγκεκριμένα, το ROC-AUC ανήλθε σε 0.80, υποδηλώνοντας καλή διακριτική ικανότητα μεταξύ ασθενών υψηλού και μη υψηλού κόστους, ενώ το PR-AUC (0.41) ήταν σημαντικά υψηλότερο από το baseline ποσοστό του θετικού γεγονότος (~10%), επιβεβαιώνοντας ουσιαστική βελτίωση έναντι τυχαίας πρόβλεψης. Το μοντέλο κατάφερε να εντοπίσει περίπου το 62% των μελλοντικών υψηλού κόστους ασθενών (recall), με μέτρια ακρίβεια (precision 0.32), γεγονός που αντανακλά τον κλασικό συμβιβασμό μεταξύ ευαισθησίας και ειδικότητας σε σενάρια risk stratification. Συνολικά, τα αποτελέσματα κρίνονται στατιστικά εύλογα και μεθοδολογικά ορθά, χωρίς ενδείξεις διαρροής πληροφορίας (data leakage), παρέχοντας ένα αξιόπιστο baseline μοντέλο πρόβλεψης κόστους σε πραγματικό περιβάλλον υγειονομικής ανάλυσης.